In [ ]:
# imports
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

In [ ]:
# load clean data (no pickle needed)
X = np.load("../data/processed/X.npy")
y = np.load("../data/processed/y.npy")

In [ ]:
# check dtype and shape
X.dtype, y.dtype, X.shape, y.shape

In [ ]:
# use a subset (for faster training)
X_small = X[:200000]
y_small = y[:200000]

In [ ]:
# convert to numeric (remove commas)
X_small = np.array([
    [float(str(value).replace(',', '')) for value in row]
    for row in X_small
], dtype=np.float32)

In [ ]:
# split again
X_train, X_test, y_train, y_test = train_test_split(
    X_small, y_small, test_size=0.2, random_state=42
)

In [ ]:
# normalization
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

std[std == 0] = 1

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

In [ ]:
# compute positive class weight
pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()

pos_weight = torch.tensor(pos_weight, dtype=torch.float32)

pos_weight

In [ ]:
# convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [ ]:
# check shapes
X_train.shape, y_train.shape

In [ ]:
# define model
class LaneChangeModel(nn.Module):
    def __init__(self):
        super(LaneChangeModel, self).__init__()
        
        self.model = nn.Sequential(
            nn.Linear(75, 64),   # input → hidden
            nn.ReLU(),
            
            nn.Linear(64, 32),   # hidden → hidden
            nn.ReLU(),
            
            nn.Linear(32, 1),    # output
            # nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.model(x)

In [ ]:
# create model instance
model = LaneChangeModel()

model

In [ ]:
# weighted loss
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [ ]:
# optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# training loop
epochs = 5

for epoch in range(epochs):
    
    model.train()
    
    # forward pass
    outputs = model(X_train).squeeze()
    
    # compute loss
    loss = criterion(outputs, y_train)
    
    # backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

In [ ]:
# get probabilities (apply sigmoid manually now)
model.eval()

with torch.no_grad():
    outputs = torch.sigmoid(model(X_test)).squeeze()

In [ ]:
# ground truth
y_true = y_test.numpy()

In [ ]:
# default threshold
threshold = 0.5
preds = (outputs >= threshold).float().numpy()

In [ ]:
# compute accuracy
accuracy = (preds == y_test).float().mean()

accuracy.item()

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(y_true, preds)
recall = recall_score(y_true, preds)
f1 = f1_score(y_true, preds)

precision, recall, f1

In [ ]:
# threshold search
import numpy as np

thresholds = np.linspace(0.1, 0.9, 9)

results = []

for t in thresholds:
    preds = (outputs >= t).float().numpy()
    
    precision = precision_score(y_true, preds)
    recall = recall_score(y_true, preds)
    f1 = f1_score(y_true, preds)
    
    results.append((t, precision, recall, f1))

results

In [ ]:
# best threshold based on F1
best = max(results, key=lambda x: x[3])
best

In [ ]:
from sklearn.metrics import confusion_matrix

best_threshold = best[0]
preds = (outputs >= best_threshold).float().numpy()

cm = confusion_matrix(y_true, preds)
cm

In [ ]:
# confusion matrix heatmap
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure()
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=["No Change", "Lane Change"],
            yticklabels=["No Change", "Lane Change"])

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()